In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATASET_DIR = Path('/content/drive/MyDrive/Offical Project.v8i.coco')
OUTPUT_DIR = Path('/content/drive/MyDrive/hazard_model_runs')
MANIFEST_PATH = OUTPUT_DIR / 'splits.csv'

VEGETATION_CATEGORY = 'vegetation'
POWERLINE_CATEGORY = 'Power Line'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
import json
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

@dataclass(frozen=True)
class Box:
    class_id: int
    x1: float
    y1: float
    x2: float
    y2: float

@dataclass(frozen=True)
class Row:
    path: Path
    score: float
    label: int
    split: str

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
DANGER_DISTANCE = 0.03
SOFTNESS = 0.015
LABEL_THRESHOLD = 0.50

def resolve_category_id(categories, value):
    try:
        return int(value)
    except ValueError:
        pass

    normalized = str(value).strip().lower()
    matches = [
        int(category['id'])
        for category in categories
        if str(category.get('name', '')).strip().lower() == normalized
    ]
    if len(matches) == 1:
        return matches[0]

    available = ', '.join(f"{category['id']}:{category['name']}" for category in categories)
    raise ValueError(f'Could not resolve category {value!r}. Available categories: {available}')

def coco_box(annotation, image):
    width = max(float(image['width']), 1.0)
    height = max(float(image['height']), 1.0)
    x, y, box_width, box_height = [float(value) for value in annotation['bbox']]
    return Box(
        class_id=int(annotation['category_id']),
        x1=max(0.0, x / width),
        y1=max(0.0, y / height),
        x2=min(1.0, (x + box_width) / width),
        y2=min(1.0, (y + box_height) / height),
    )

def normalized_box_gap(a, b):
    horizontal_gap = max(a.x1 - b.x2, b.x1 - a.x2, 0.0)
    vertical_gap = max(a.y1 - b.y2, b.y1 - a.y2, 0.0)
    return math.sqrt(horizontal_gap**2 + vertical_gap**2)

def compute_hazard_score(boxes, vegetation_class, powerline_class, danger_distance, softness):
    vegetation_boxes = [box for box in boxes if box.class_id == vegetation_class]
    powerline_boxes = [box for box in boxes if box.class_id == powerline_class]
    if not vegetation_boxes or not powerline_boxes:
        return 0.0

    closest_gap = min(
        normalized_box_gap(vegetation, powerline)
        for vegetation in vegetation_boxes
        for powerline in powerline_boxes
    )
    raw = (danger_distance - closest_gap) / max(softness, 1e-6)
    score = 1.0 / (1.0 + math.exp(-raw))
    return float(max(0.0, min(1.0, score)))

In [ ]:
def build_coco_manifest(
    dataset_dir,
    output_path,
    vegetation_category='vegetation',
    powerline_category='Power Line',
    danger_distance=0.03,
    softness=0.015,
    label_threshold=0.5,
):
    dataset_dir = Path(dataset_dir)
    output_path = Path(output_path)
    rows = []
    category_names = {}

    for split_dir_name, split_name in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
        annotation_path = dataset_dir / split_dir_name / '_annotations.coco.json'
        if not annotation_path.exists():
            print('Skipping missing split:', annotation_path)
            continue

        data = json.loads(annotation_path.read_text())
        categories = data.get('categories', [])
        vegetation_class = resolve_category_id(categories, str(vegetation_category))
        powerline_class = resolve_category_id(categories, str(powerline_category))
        category_names.update({int(category['id']): category['name'] for category in categories})

        images = {int(image['id']): image for image in data.get('images', [])}
        annotations_by_image = defaultdict(list)
        for annotation in data.get('annotations', []):
            annotations_by_image[int(annotation['image_id'])].append(annotation)

        for image_id, image in sorted(images.items()):
            boxes = [coco_box(annotation, image) for annotation in annotations_by_image[image_id]]
            score = compute_hazard_score(
                boxes,
                vegetation_class,
                powerline_class,
                danger_distance,
                softness,
            )
            rows.append({
                'path': str(dataset_dir / split_dir_name / image['file_name']),
                'label_path': str(annotation_path),
                'hazard_score': f'{score:.6f}',
                'label': str(int(score >= label_threshold)),
                'split': split_name,
            })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['path', 'label_path', 'hazard_score', 'label', 'split'])
        writer.writeheader()
        writer.writerows(rows)

    summary = Counter((row['split'], row['label']) for row in rows)
    print('Wrote manifest:', output_path)
    print('Categories:', category_names)
    print('Images:', len(rows))
    for split in ['train', 'val', 'test']:
        print(f"{split}: safe={summary[(split, '0')]} hazardous={summary[(split, '1')]}")

build_coco_manifest(
    DATASET_DIR,
    MANIFEST_PATH,
    vegetation_category=VEGETATION_CATEGORY,
    powerline_category=POWERLINE_CATEGORY,
    danger_distance=DANGER_DISTANCE,
    softness=SOFTNESS,
    label_threshold=LABEL_THRESHOLD,
)

In [ ]:
class HazardDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row.path) as image:
            image = image.convert('RGB')
        return self.transform(image), torch.tensor(row.score, dtype=torch.float32), row.label, str(row.path)

def read_manifest(path, split):
    rows = []
    with Path(path).open() as handle:
        for raw in csv.DictReader(handle):
            if raw['split'] == split:
                rows.append(Row(Path(raw['path']), float(raw['hazard_score']), int(raw['label']), raw['split']))
    return rows

def build_transforms(image_size):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.12),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    return train_tf, eval_tf

def build_model(model_name='convnext_tiny'):
    if model_name == 'convnext_tiny':
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, 1)
        return model
    if model_name == 'vit_b_16':
        weights = models.ViT_B_16_Weights.DEFAULT
        model = models.vit_b_16(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, 1)
        return model
    raise ValueError(f'Unknown model: {model_name}')

def make_train_loader(dataset, rows, batch_size, num_workers=2, balanced_samples=None):
    counts = Counter(row.label for row in rows)
    if len(counts) < 2:
        return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    weights = [1.0 / counts[row.label] for row in rows]
    if balanced_samples is None:
        balanced_samples = 2 * min(counts.values())
    sampler = WeightedRandomSampler(weights, num_samples=balanced_samples, replacement=True)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers)

def make_eval_loader(dataset, batch_size, num_workers=2):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
MODEL_NAME = 'convnext_tiny' 
IMAGE_SIZE = 224
EPOCHS = 8
BATCH_SIZE = 16
LR = 3e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

RUN_DIR = OUTPUT_DIR / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

def hazard_metrics(labels, targets, probs, threshold=0.5):
    preds = [1 if prob >= threshold else 0 for prob in probs]
    metrics = {
        'mae': mean_absolute_error(targets, probs),
        'accuracy': accuracy_score(labels, preds),
        'precision_hazard': precision_score(labels, preds, zero_division=0),
        'recall_hazard': recall_score(labels, preds, zero_division=0),
        'f1_hazard': f1_score(labels, preds, zero_division=0),
        'confusion_matrix': confusion_matrix(labels, preds, labels=[0, 1]).tolist(),
    }
    if len(set(labels)) == 2:
        metrics['roc_auc'] = roc_auc_score(labels, probs)
        metrics['average_precision'] = average_precision_score(labels, probs)
    else:
        metrics['roc_auc'] = None
        metrics['average_precision'] = None
    return metrics

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    labels_all, targets_all, probs_all = [], [], []
    total_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()

    for images, scores, labels, _ in loader:
        images = images.to(device)
        scores = scores.to(device)
        logits = model(images).squeeze(1)
        loss = criterion(logits, scores)
        probs = torch.sigmoid(logits)
        total_loss += loss.item() * scores.size(0)
        labels_all.extend([int(label) for label in labels])
        targets_all.extend(scores.cpu().tolist())
        probs_all.extend(probs.cpu().tolist())

    metrics = hazard_metrics(labels_all, targets_all, probs_all)
    metrics['loss'] = total_loss / max(1, len(labels_all))
    return metrics

def train_model():
    set_seed(42)
    train_rows = read_manifest(MANIFEST_PATH, 'train')
    val_rows = read_manifest(MANIFEST_PATH, 'val')
    train_tf, eval_tf = build_transforms(IMAGE_SIZE)
    train_dataset = HazardDataset(train_rows, train_tf)
    val_dataset = HazardDataset(val_rows, eval_tf)
    train_loader = make_train_loader(train_dataset, train_rows, BATCH_SIZE, NUM_WORKERS)
    val_loader = make_eval_loader(val_dataset, BATCH_SIZE, NUM_WORKERS)

    device = torch.device(device_name())
    model = build_model(MODEL_NAME).to(device)

    counts = Counter(row.label for row in train_rows)
    pos_weight = None
    if counts[0] and counts[1]:
        pos_weight = torch.tensor([counts[0] / counts[1]], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_score = -1.0
    history = []
    print(f'Training {MODEL_NAME} on {device} -> {RUN_DIR}')
    print(f'Train rows: {len(train_rows)} | Val rows: {len(val_rows)}')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        started = time.time()
        running_loss = 0.0
        seen = 0

        for images, scores, _, _ in train_loader:
            images = images.to(device)
            scores = scores.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images).squeeze(1)
            loss = criterion(logits, scores)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * scores.size(0)
            seen += scores.size(0)

        scheduler.step()
        val_metrics = evaluate(model, val_loader, device)
        train_loss = running_loss / max(1, seen)
        record = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val': val_metrics,
            'seconds': round(time.time() - started, 1),
        }
        history.append(record)

        auc_text = 'nan' if val_metrics['roc_auc'] is None else f"{val_metrics['roc_auc']:.4f}"
        print(
            f"epoch {epoch}/{EPOCHS} "
            f"train_loss={train_loss:.4f} "
            f"val_f1={val_metrics['f1_hazard']:.4f} "
            f"val_recall={val_metrics['recall_hazard']:.4f} "
            f"val_precision={val_metrics['precision_hazard']:.4f} "
            f"val_mae={val_metrics['mae']:.4f} "
            f"val_auc={auc_text}"
        )

        selection_score = val_metrics['f1_hazard'] - val_metrics['mae']
        if selection_score > best_score:
            best_score = selection_score
            torch.save({
                'model': MODEL_NAME,
                'state_dict': model.state_dict(),
                'image_size': IMAGE_SIZE,
                'val_metrics': val_metrics,
            }, RUN_DIR / 'best.pt')

        with (RUN_DIR / 'history.json').open('w') as handle:
            json.dump(history, handle, indent=2)

    print('Best checkpoint:', RUN_DIR / 'best.pt')
    return model

model = train_model()

In [ ]:
def load_checkpoint(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model = build_model(checkpoint['model'])
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model, checkpoint

checkpoint_path = RUN_DIR / 'best.pt'
model, checkpoint = load_checkpoint(checkpoint_path)
device = torch.device(device_name())
model = model.to(device)

_, eval_tf = build_transforms(checkpoint.get('image_size', IMAGE_SIZE))
test_rows = read_manifest(MANIFEST_PATH, 'test')
test_dataset = HazardDataset(test_rows, eval_tf)
test_loader = make_eval_loader(test_dataset, batch_size=32, num_workers=NUM_WORKERS)

test_metrics = evaluate(model, test_loader, device)
with (RUN_DIR / 'test_metrics.json').open('w') as handle:
    json.dump(test_metrics, handle, indent=2)

print(json.dumps(test_metrics, indent=2))
print('Wrote:', RUN_DIR / 'test_metrics.json')

In [ ]:
@torch.no_grad()
def predict_image(image_path, checkpoint_path=RUN_DIR / 'best.pt', threshold=0.5):
    model, checkpoint = load_checkpoint(checkpoint_path)
    device = torch.device(device_name())
    model = model.to(device)
    _, eval_tf = build_transforms(checkpoint.get('image_size', IMAGE_SIZE))

    image_path = Path(image_path)
    with Image.open(image_path) as image:
        image = image.convert('RGB')
    tensor = eval_tf(image).unsqueeze(0).to(device)
    score = torch.sigmoid(model(tensor).squeeze()).item()
    return {
        'image': str(image_path),
        'hazard_score': round(float(score), 6),
        'hazard_label': int(score >= threshold),
        'threshold': threshold,
    }

sample_image = test_rows[0].path
prediction = predict_image(sample_image)
print(json.dumps(prediction, indent=2))

In [ ]:
from google.colab import files
files.download(str(RUN_DIR / 'best.pt'))